# Week 01 · Day 03 — RAG Basics

**Topics:** Chunking strategies · ChromaDB ingestion · End-to-end RAG pipeline


In [ ]:
%pip install -q openai chromadb langchain-text-splitters

In [ ]:
import os
from openai import OpenAI
import chromadb

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma = chromadb.Client()  # in-memory; use chromadb.PersistentClient(path="./db") to persist

## 1 · Chunking Strategies

LLMs can't read an entire document at once. Chunking splits documents into pieces that fit within a retrieval context. The right chunk size depends on your documents and queries.

In [ ]:
# Strategy 1: Fixed-size chunking
def fixed_chunk(text: str, size: int = 400, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

sample_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that retrieves relevant 
information from a knowledge base before generating a response. This grounds the 
model's output in factual, up-to-date information rather than relying solely on 
what was learned during training.

The RAG pipeline has two phases: ingestion and query. During ingestion, documents 
are chunked, embedded, and stored in a vector database. During query, the user's 
question is embedded and used to retrieve the most relevant chunks, which are then 
passed to the language model as context.

RAG is preferred over fine-tuning when the knowledge base changes frequently, 
when you need source citations, or when training compute is limited. Fine-tuning 
is better for teaching the model a new skill or style, not new factual knowledge.
""".strip()

fixed_chunks = fixed_chunk(sample_text, size=300, overlap=50)
print(f"Fixed chunking: {len(fixed_chunks)} chunks")
for i, c in enumerate(fixed_chunks):
    print(f"  Chunk {i}: {len(c)} chars — {c[:60]}...")

In [ ]:
# Strategy 2: Recursive character text splitting (smarter — respects sentence/paragraph boundaries)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries each separator in order
)

recursive_chunks = splitter.split_text(sample_text)
print(f"Recursive chunking: {len(recursive_chunks)} chunks")
for i, c in enumerate(recursive_chunks):
    print(f"  Chunk {i}: {len(c)} chars — {c[:60]}...")

## 2 · Embedding Chunks and Storing in ChromaDB

In [ ]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    r = client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in r.data]

# Create a ChromaDB collection
collection = chroma.get_or_create_collection("rag_demo")

# Ingest the recursive chunks
embeddings = get_embeddings(recursive_chunks)

collection.add(
    documents=recursive_chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(recursive_chunks))],
    metadatas=[{"source": "rag_intro", "chunk_index": i} for i in range(len(recursive_chunks))],
)

print(f"Stored {collection.count()} chunks in ChromaDB")

## 3 · Retrieval

In [ ]:
def retrieve(query: str, n_results: int = 3) -> list[str]:
    query_embedding = get_embeddings([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )
    return results["documents"][0]

query = "When should I use RAG instead of fine-tuning?"
chunks = retrieve(query)

print(f"Retrieved {len(chunks)} chunks for: '{query}'\n")
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk[:100]}...")

## 4 · End-to-End RAG Pipeline

Combine retrieval and generation into a single function.

In [ ]:
def rag_answer(user_query: str, n_results: int = 3) -> str:
    # Retrieve relevant chunks
    retrieved_chunks = retrieve(user_query, n_results=n_results)
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(retrieved_chunks))

    # Generate answer grounded in retrieved context
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question using only the provided context. "
                    "If the answer is not in the context, say 'I don't have that information.' "
                    "Cite the context number [1], [2], etc. when using information from it."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {user_query}",
            },
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content

questions = [
    "What are the two phases of the RAG pipeline?",
    "When should I fine-tune instead of using RAG?",
    "What is the capital of Japan?",  # out of context
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print()

## 5 · Adding More Documents (Realistic Ingestion)

In production, your ingestion pipeline runs offline and populates the vector DB before any queries arrive.

In [ ]:
additional_docs = [
    {"id": "doc_python", "text": "Python was created by Guido van Rossum and released in 1991. It emphasizes code readability and simplicity.", "source": "python_intro"},
    {"id": "doc_llm", "text": "Large language models are trained on massive text corpora using transformer architectures. They learn statistical patterns to predict the next token.", "source": "llm_intro"},
    {"id": "doc_vector", "text": "Vector databases store embeddings and support approximate nearest neighbor search. They are the backbone of any RAG system.", "source": "vector_db"},
]

texts = [d["text"] for d in additional_docs]
embs = get_embeddings(texts)

collection.add(
    documents=texts,
    embeddings=embs,
    ids=[d["id"] for d in additional_docs],
    metadatas=[{"source": d["source"]} for d in additional_docs],
)

print(f"Collection now has {collection.count()} documents")
print()

# Query the expanded collection
print(rag_answer("What is a vector database used for?"))

## Summary

- **Fixed chunking**: simple but can split mid-sentence. Good for homogeneous text.
- **Recursive splitting**: smarter — tries `\n\n`, then `\n`, then `.` etc. Preferred.
- **ChromaDB**: in-memory by default; `PersistentClient(path=...)` for persistence.
- **RAG pipeline**: retrieve chunks → build context string → generate with system constraint.
- Always instruct the model to say "I don't know" when context is insufficient.

**Next:** [Vector Databases](week01-day03-vector-databases.ipynb)